### Import packages

In [ ]:
from enum import Enum
import cvxpy as cp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.ticker import MaxNLocator
import pandas as pd

In [ ]:
class Season(Enum):
	WINTER = 'jan'
	SPRING = 'apr'
	SUMMER = 'jul'
	AUTUMN = 'oct'

### Settings
Edit these to scale up the time horizon or facility size, or change season.

In [ ]:
DAYS = 5					# number of days to model
I = 1						# number of containers to model
season = Season.AUTUMN		# season for RTM prices

### Define parameters

In [ ]:
# !!! OPERATIONAL PARAMETERS - DO NOT CHANGE !!!
DELTA_t = 1.0		# hr
HOURS_PER_DAY = 24
N = DAYS * HOURS_PER_DAY
SOC_MIN = 0.05		# dimensionless
SOC_MAX = 0.95		# dimensionless
Q_NEW = 3.916		# MWh, new container capacity
P_MAX = 0.979		# MW, hard limit
T_MIN = -30			# °C
T_REF = 25			# °C
T_MAX = 50			# °C
ETA = 0.968			# dimensionless

In [ ]:
# FLEXIBLE MODEL PARAMS - CAN CHANGE FOR SENSITIVITY ANALYSIS
ALPHA = 2.61e-5		# MWh capacity lost per MWh throughput
BETA = 2.5			# °C / MWh throughput
GAMMA = 3.0			# °C / hr
KAPPA = 0.04		# / °C
RHO = 19500			# $ / MWh capacity lost (opportunity)
SIGMA = 241000		# $ / MWh capacity lost (replacement)
A = 0.02			# dimensionless
B = 0.02			# dimensionless

In [ ]:
# Day starting values for containers, spaced equally in range
SOH_0 = np.linspace(0.98, 0.82, I)	# dimensionless
SOC_0 = np.linspace(0.8, 0.2, I)	# dimensionless

# Day ending values
SOC_N = SOC_0

### Generate market price trajectory

In [ ]:
expected_prices_df = pd.read_csv("./data/raw/DAM/seasonal_stats.csv")
LAMBDA_DAILY = expected_prices_df[f'{season.value}_mean'].to_numpy().reshape(-1, 1)
LAMBDA = np.tile(LAMBDA_DAILY, (DAYS, 1))

### Define state & decision variables

In [ ]:
# states: k = 0 -> N, i = 1 -> 2
E = cp.Variable((N + 1, I), nonneg=True)
Q = cp.Variable((N + 1, I), nonneg=True)
T = cp.Variable((N + 1, I), nonneg=True)

# decisions: k = 0 -> N-1, i = 1 -> 2
c = cp.Variable((N, I), nonneg=True)
d = cp.Variable((N, I), nonneg=True)
u = cp.Variable((N, I), nonneg=True)

### Define constraints

In [ ]:
phi = cp.Parameter((N,I), nonneg=True)
delta_Q = ALPHA * cp.multiply(phi, c + d) * DELTA_t

constraints = [
	# Initial conditions
	Q[0, :] == Q_NEW * SOH_0,
	E[0, :] == cp.multiply(Q[0, :], SOC_0),
	T[0, :] == T_REF,
	# Dynamics - see below for capacity dynamics
	Q[1:, :] == Q[:-1, :] - delta_Q,
	E[1:, :] == E[:-1, :] + (ETA * c * DELTA_t) - (d * DELTA_t / ETA),
	T[1:, :] == T[:-1, :] + (BETA * (c + d) * DELTA_t) - (GAMMA * u * DELTA_t),
	# Boundary conditions
	# Q >= 0, # variable created as non-neg=True
	E >= Q * SOC_MIN,
	E <= Q * SOC_MAX,
	T >= T_MIN,
	T <= T_MAX,
	# c >= 0, # variable created as non-neg=True
	c <= P_MAX,
	# d >= 0, # variable created as non-neg=True
	d <= P_MAX,
	# u >= 0, # variable created as non-neg=True
	u <= 1,
	# Terminal conditions
	E[N, :] >= cp.multiply(Q[N, :], SOC_N)
]

### Define objective function and problem

In [ ]:
arbitrage_cost = cp.sum(cp.multiply(LAMBDA, c - d) * DELTA_t)
operating_cost = cp.sum(cp.multiply(LAMBDA, A + B * u) * P_MAX * DELTA_t)
opportunity_plus_replacement_cost = cp.sum((RHO + SIGMA) * delta_Q)
objective = cp.Minimize(arbitrage_cost + opportunity_plus_replacement_cost + operating_cost)
problem = cp.Problem(objective, constraints)

### Successive convex programming

Unfortunately, our temperature-dependent state transition function for capacity is non-convex. 

The following workaround solves the problem iteratively instead:
1. Temperature remains a regular state variable. But _only when calculating the temperature degradation factor_ $\phi(k)$, we initially assume the temperature is $T_\text{ref}$ the whole time.
2. We solve the problem. Temperature changes based on our optimized decisions, but the _degradation_ is as if it stayed static.
3. We plug in our solved values for temperature as a _static_ guess in order to calculate another iteration of $\phi(k)$, and repeat.
4. This converges in just three loops (I tried 10 initially). That is, the temperature state trajectory stops changing with each iteration.

Essentially, while temperature is a state variable, we use the previous iteration's trajectory for temperature as a fixed parameter to calculate $\phi(k)$, making the problem convex.

In [ ]:
T_est = np.full((N, I), T_REF)

for iteration in range(4):
	print(f"Iteration {iteration} starting...")

	phi.value = 1 + KAPPA * np.maximum(0, T_est - T_REF)

	problem.solve(
		solver=cp.ECOS,
		verbose=True
	)

	if problem.status == "optimal":
		T_est = T.value[:-1, :]
		print(f"Iteration {iteration}: Total Cost = {problem.value:.2f}")
	else:
		print("Solver failed to find an optimal solution.")
		break


### Plot

In [ ]:
color_map = cm.get_cmap('tab10', I)
markers = ['o', 's', '^', 'D', 'x', '+', 'v', '<', '>', 'p', '*', 'h']

fig, axes = plt.subplots(5, figsize=(10, 25))
K = np.arange(N + 1)

for i in range(I):
	axes[0].plot(K, E.value[:, i],
		color=color_map(i),
		linewidth=2,
		label=f'Container {i+1}',
		marker=markers[i % len(markers)],
		markersize=4
	)

axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Stored Energy (MWh)', fontweight='bold')
axes[0].set_ylim(0, Q_NEW + 1)
axes[0].set_title('BESS Energy Dispatch vs. Market Price')
axes[0].legend(loc='upper left', bbox_to_anchor=(1.1, 1))

ax2 = axes[0].twinx()
ax2.set_ylabel('RTM Price ($/MWh)', fontweight='bold')
ax2.step(K[:-1], LAMBDA, where='post', color='black', alpha=0.3, label='Market Price')
ax2.tick_params(axis='y')

for i in range(I):
	axes[1].plot(K, Q.value[:, i] - Q.value[0, i],
		label=f"Container {i+1}",
		color=color_map(i),
		marker=markers[i % len(markers)],
		markersize=4
	)

axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Change in Capacity (MWh)', fontweight='bold')
axes[1].set_title(r"Degradation: Change in Container Capacity ($\Delta Q$)")
axes[1].legend(loc='upper left', bbox_to_anchor=(1.1, 1))

for i in range(I):
	axes[2].plot(K, T.value[:, i],
		label=f"Container {i+1}",
		color=color_map(i),
		marker=markers[i % len(markers)],
		markersize=4
	)

axes[2].set_xlabel('Hour of Day')
axes[2].set_ylabel('Temperature ($T$)', fontweight='bold')
axes[2].set_title("Container Thermal State (°C)")
axes[2].legend(loc='upper left', bbox_to_anchor=(1.1, 1))

for i in range(I):
	axes[3].plot(K[:-1], c.value[:, i],
		marker='^', color=color_map(i),
		label=f"C{i+1} Charge", markersize=4
	)
	axes[3].plot(K[:-1], -d.value[:, i],
		marker='v', color=color_map(i), linestyle='--',
		label=f"C{i+1} Discharge", markersize=4
	)

axes[3].axhline(0, color='black', linewidth=0.8, alpha=0.5)
axes[3].set_xlabel("Hour of Day")
axes[3].set_ylabel("Power Setpoint (MW)")
axes[3].set_title("Power Setpoints (Charge vs. Discharge)")
axes[3].legend(loc='upper left', bbox_to_anchor=(1.1, 1))

for i in range(I):
	axes[4].plot(K[:-1], u.value[:, i],
		label=f"Container {i+1}",
		color=color_map(i),
		marker=markers[i % len(markers)],
		markersize=4)

axes[4].set_xlabel('Hour of Day')
axes[4].set_ylabel(r'Cooling effort $u \in [0,1]$', fontweight='bold')
axes[4].set_title("HVAC/Cooling Effort")
axes[4].set_ylim(-0.05, 1.05)
axes[4].legend(loc='upper left', bbox_to_anchor=(1.1, 1))

for ax in axes:
	ax.grid(True, linestyle='--', alpha=0.6)
	ax.xaxis.set_major_locator(MaxNLocator(nbins=25, integer=True))
	ax.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()